# Setup

In [1]:
import time
from functools import partial
from itertools import product
from typing import NamedTuple

import jax
import jax.numpy as jnp
import jaxopt
import numpy as np
import pandas as pd
import scipy.optimize
from jax import block_until_ready, jit, lax, pmap, random, vmap
from scipy.optimize import minimize

In [2]:
class Algorithm(NamedTuple):
    name: str
    function: callable
    parameters: dict

class Benchmark(NamedTuple):
    name: str
    function: callable
    lower_bound: float
    upper_bound: float

class Result(NamedTuple):
    dimension: int
    benchmark: str
    algorithm: str
    best_fitness: float
    best_position: np.ndarray | jax.Array
    fitness_history: list[list[float]]
    execution_time: float

class SwarmState(NamedTuple):
    positions: np.ndarray
    velocities: np.ndarray
    personal_best_positions: np.ndarray
    personal_best_fitness: np.ndarray
    global_best_position: np.ndarray
    global_best_fitness: np.ndarray
    rng: np.random.Generator
    fitness_history: np.ndarray

class JaxSwarmState(NamedTuple):
    positions: jax.Array
    velocities: jax.Array
    personal_best_positions: jax.Array
    personal_best_fitness: jax.Array
    global_best_position: jax.Array
    global_best_fitness: jax.Array
    rng: random.PRNGKey

class GradientState(NamedTuple):
    current_pos: jax.Array

## PSO (NumPy)

In [3]:
def pso_numpy(
    seed: int,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    inertia_weight: float,
    cognitive_coeff: float,
    social_coeff: float,
    max_iterations: int,
) -> tuple:
    rng = np.random.default_rng(seed)

    initial_positions = rng.uniform(
        lower_bound, upper_bound, (n_particles, n_dimensions),
    )
    initial_velocities = np.zeros((n_particles, n_dimensions))
    initial_fitness = np.array(
        [objective_function(position) for position in initial_positions],
    )

    best_particle_idx = np.argmin(initial_fitness)
    global_best_position = initial_positions[best_particle_idx]
    global_best_fitness = initial_fitness[best_particle_idx]

    fitness_history = np.zeros(max_iterations)

    swarm_state = SwarmState(
        positions=initial_positions,
        velocities=initial_velocities,
        personal_best_positions=initial_positions,
        personal_best_fitness=initial_fitness,
        global_best_position=global_best_position,
        global_best_fitness=global_best_fitness,
        rng=rng,
        fitness_history=fitness_history,
    )

    for iteration in range(max_iterations):
        r1 = swarm_state.rng.random((n_particles, n_dimensions))
        r2 = swarm_state.rng.random((n_particles, n_dimensions))

        inertia_term = inertia_weight * swarm_state.velocities
        cognitive_term = (
            cognitive_coeff * r1 * (
                swarm_state.personal_best_positions - swarm_state.positions
            )
        )
        social_term = (
            social_coeff * r2 * (
                swarm_state.global_best_position - swarm_state.positions
            )
        )

        updated_velocities = inertia_term + cognitive_term + social_term
        updated_positions = swarm_state.positions + updated_velocities
        updated_positions = np.clip(updated_positions, lower_bound, upper_bound)

        updated_fitness = np.array(
            [objective_function(position) for position in updated_positions],
        )

        personal_improved = updated_fitness < swarm_state.personal_best_fitness
        improvement_mask = personal_improved[:, None]
        updated_personal_best_positions = np.where(
            improvement_mask,
            updated_positions,
            swarm_state.personal_best_positions,
        )
        updated_personal_best_fitness = np.where(
            personal_improved,
            updated_fitness,
            swarm_state.personal_best_fitness,
        )

        global_best_candidate_idx = np.argmin(updated_personal_best_fitness)
        global_best_candidate_fitness = updated_personal_best_fitness[
            global_best_candidate_idx
        ]
        global_improved = (
            global_best_candidate_fitness < swarm_state.global_best_fitness
        )

        updated_global_best_position = np.where(
            global_improved,
            updated_personal_best_positions[global_best_candidate_idx],
            swarm_state.global_best_position,
        )
        updated_global_best_fitness = np.where(
            global_improved,
            global_best_candidate_fitness,
            swarm_state.global_best_fitness,
        )

        updated_fitness_history = swarm_state.fitness_history
        updated_fitness_history[iteration] = updated_global_best_fitness

        swarm_state = SwarmState(
            positions=updated_positions,
            velocities=updated_velocities,
            personal_best_positions=updated_personal_best_positions,
            personal_best_fitness=updated_personal_best_fitness,
            global_best_position=updated_global_best_position,
            global_best_fitness=updated_global_best_fitness,
            rng=swarm_state.rng,
            fitness_history=updated_fitness_history,
        )

    return (
        swarm_state.global_best_position,
        swarm_state.global_best_fitness,
        swarm_state.fitness_history,
    )

## PSO (JAX)

In [4]:
@partial(
    jit,
    static_argnames=(
        "objective_function",
        "lower_bound",
        "upper_bound",
        "n_dimensions",
        "n_particles",
        "max_iterations",
    ),
)
def pso_jax(
    seed: random.PRNGKey,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    inertia_weight: float,
    cognitive_coeff: float,
    social_coeff: float,
    max_iterations: int,
) -> tuple:
    position_key, velocity_key, state_key = random.split(seed, 3)

    search_range = upper_bound - lower_bound
    velocity_scale = 0.1
    velocity_limit = search_range * velocity_scale

    initial_positions = random.uniform(
        position_key,
        (n_particles, n_dimensions),
        minval=lower_bound,
        maxval=upper_bound,
    )
    initial_velocities = random.uniform(
        velocity_key,
        (n_particles, n_dimensions),
        minval=-velocity_limit,
        maxval=velocity_limit,
    )
    initial_fitness = vmap(objective_function)(initial_positions)

    best_particle_idx = jnp.argmin(initial_fitness)
    global_best_position = initial_positions[best_particle_idx]
    global_best_fitness = initial_fitness[best_particle_idx]

    swarm_state = JaxSwarmState(
        positions=initial_positions,
        velocities=initial_velocities,
        personal_best_positions=initial_positions,
        personal_best_fitness=initial_fitness,
        global_best_position=global_best_position,
        global_best_fitness=global_best_fitness,
        rng=state_key,
    )

    def update_step(swarm_state: JaxSwarmState, _: any) -> tuple:
        cognitive_key, social_key, next_key = random.split(swarm_state.rng, 3)
        r1 = random.uniform(cognitive_key, (n_particles, n_dimensions))
        r2 = random.uniform(social_key, (n_particles, n_dimensions))

        inertia_term = inertia_weight * swarm_state.velocities
        cognitive_term = (
            cognitive_coeff
            * r1
            * (swarm_state.personal_best_positions - swarm_state.positions)
        )
        social_term = (
            social_coeff
            * r2
            * (swarm_state.global_best_position - swarm_state.positions)
        )

        updated_velocities = inertia_term + cognitive_term + social_term
        updated_positions = swarm_state.positions + updated_velocities
        updated_positions = jnp.clip(updated_positions, lower_bound, upper_bound)

        updated_fitness = vmap(objective_function)(updated_positions)

        personal_improved = updated_fitness < swarm_state.personal_best_fitness

        updated_personal_best_positions = jnp.where(
            personal_improved[:, None],
            updated_positions,
            swarm_state.personal_best_positions,
        )
        updated_personal_best_fitness = jnp.where(
            personal_improved,
            updated_fitness,
            swarm_state.personal_best_fitness,
        )

        global_best_candidate_idx = jnp.argmin(updated_personal_best_fitness)
        global_best_candidate_position = updated_personal_best_positions[
            global_best_candidate_idx
        ]
        global_best_candidate_fitness = updated_personal_best_fitness[
            global_best_candidate_idx
        ]

        global_improved = (
            global_best_candidate_fitness < swarm_state.global_best_fitness
        )

        updated_global_best_position = jnp.where(
            global_improved,
            global_best_candidate_position,
            swarm_state.global_best_position,
        )
        updated_global_best_fitness = jnp.where(
            global_improved,
            global_best_candidate_fitness,
            swarm_state.global_best_fitness,
        )

        next_state = JaxSwarmState(
            positions=updated_positions,
            velocities=updated_velocities,
            personal_best_positions=updated_personal_best_positions,
            personal_best_fitness=updated_personal_best_fitness,
            global_best_position=updated_global_best_position,
            global_best_fitness=updated_global_best_fitness,
            rng=next_key,
        )

        return next_state, updated_global_best_fitness

    final_state, fitness_history = lax.scan(
        update_step,
        swarm_state,
        jnp.arange(max_iterations),
    )

    return (
        final_state.global_best_position,
        final_state.global_best_fitness,
        fitness_history,
    )

## PSOX (NumPy + SciPy)

In [5]:
def approximate_gradient(
    func: callable,
    position: np.ndarray,
    epsilon: float = 1e-8,
) -> np.ndarray:
    grad = np.zeros_like(position)
    for i in range(len(position)):
        pos_plus = position.copy()
        pos_minus = position.copy()

        pos_plus[i] += epsilon
        pos_minus[i] -= epsilon

        grad[i] = (func(pos_plus) - func(pos_minus)) / (2 * epsilon)

    return grad


def optimize_adam_numpy(
    objective_function: callable,
    initial_position: np.ndarray,
    lower_bound: float,
    upper_bound: float,
    n_local_optimization_steps: int,
    learning_rate: float,
    beta1: float = 0.9,
    beta2: float = 0.999,
    epsilon: float = 1e-8,
    grad_epsilon: float = 1e-8,
) -> np.ndarray:
    theta = initial_position.copy()
    m = np.zeros_like(theta)
    v = np.zeros_like(theta)

    for t in range(1, n_local_optimization_steps + 1):
        g = approximate_gradient(objective_function, theta, grad_epsilon)

        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * (g ** 2)

        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)

        theta = theta - learning_rate * m_hat / (np.sqrt(v_hat) + epsilon)
        theta = np.clip(theta, lower_bound, upper_bound)

    return theta


def local_optimize(
    objective_function: callable,
    initial_position: np.ndarray,
    lower_bound: float,
    upper_bound: float,
    n_local_optimization_steps: int,
    learning_rate: float,
) -> tuple:
    best_position = initial_position.copy()
    best_fitness = objective_function(initial_position)

    bounds = [(lower_bound, upper_bound)] * len(initial_position)

    lbfgs_result = scipy.optimize.minimize(
        objective_function,
        initial_position,
        method="L-BFGS-B",
        bounds=bounds,
    )

    lbfgs_candidate = np.clip(lbfgs_result.x, lower_bound, upper_bound)
    lbfgs_fitness = objective_function(lbfgs_candidate)

    if lbfgs_fitness < best_fitness:
        best_position = lbfgs_candidate
        best_fitness = lbfgs_fitness

    cg_result = scipy.optimize.minimize(
        objective_function,
        initial_position,
        method="CG",
    )

    cg_candidate = np.clip(cg_result.x, lower_bound, upper_bound)
    cg_fitness = objective_function(cg_candidate)

    if cg_fitness < best_fitness:
        best_position = cg_candidate
        best_fitness = cg_fitness

    adam_candidate = optimize_adam_numpy(
        objective_function,
        initial_position,
        lower_bound,
        upper_bound,
        n_local_optimization_steps,
        learning_rate,
    )

    adam_fitness = objective_function(adam_candidate)

    if adam_fitness < best_fitness:
        best_position = adam_candidate
        best_fitness = adam_fitness

    return best_position, best_fitness


def psox_numpy(
    seed: int,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    inertia_weight: float,
    cognitive_coeff: float,
    social_coeff: float,
    max_iterations: int,
    local_optimization_interval: int,
    n_local_optimization_steps: int,
    learning_rate: float,
    **_: any,
) -> tuple:
    rng = np.random.default_rng(seed)

    initial_positions = rng.uniform(
        lower_bound,
        upper_bound,
        (n_particles, n_dimensions),
    )
    initial_velocities = np.zeros((n_particles, n_dimensions))
    initial_fitness = np.array(
        [objective_function(position) for position in initial_positions],
    )

    best_particle_idx = np.argmin(initial_fitness)
    global_best_position = initial_positions[best_particle_idx]
    global_best_fitness = initial_fitness[best_particle_idx]

    fitness_history = np.zeros(max_iterations)

    swarm_state = SwarmState(
        positions=initial_positions,
        velocities=initial_velocities,
        personal_best_positions=initial_positions,
        personal_best_fitness=initial_fitness,
        global_best_position=global_best_position,
        global_best_fitness=global_best_fitness,
        rng=rng,
        fitness_history=fitness_history,
    )

    for i in range(max_iterations):
        r1 = swarm_state.rng.random((n_particles, n_dimensions))
        r2 = swarm_state.rng.random((n_particles, n_dimensions))

        inertia_term = inertia_weight * swarm_state.velocities
        cognitive_term = (
            cognitive_coeff
            * r1
            * (swarm_state.personal_best_positions - swarm_state.positions)
        )
        social_term = (
            social_coeff
            * r2
            * (swarm_state.global_best_position - swarm_state.positions)
        )

        updated_velocities = inertia_term + cognitive_term + social_term
        updated_positions = swarm_state.positions + updated_velocities
        updated_positions = np.clip(updated_positions, lower_bound, upper_bound)

        updated_fitness = np.array(
            [objective_function(pos) for pos in updated_positions],
        )

        personal_improved = updated_fitness < swarm_state.personal_best_fitness
        updated_personal_best_positions = np.where(
            personal_improved[:, None],
            updated_positions,
            swarm_state.personal_best_positions,
        )
        updated_personal_best_fitness = np.where(
            personal_improved,
            updated_fitness,
            swarm_state.personal_best_fitness,
        )

        global_best_candidate_idx = np.argmin(updated_personal_best_fitness)
        global_best_candidate_fitness = updated_personal_best_fitness[
            global_best_candidate_idx
        ]
        global_improved = (
            global_best_candidate_fitness < swarm_state.global_best_fitness
        )

        global_best_candidate_position = np.where(
            global_improved,
            updated_personal_best_positions[global_best_candidate_idx],
            swarm_state.global_best_position,
        )
        updated_global_best_fitness = np.where(
            global_improved,
            global_best_candidate_fitness,
            swarm_state.global_best_fitness,
        )

        if i % local_optimization_interval == 0:
            local_opt_position, local_opt_fitness = local_optimize(
                objective_function,
                global_best_candidate_position,
                lower_bound,
                upper_bound,
                n_local_optimization_steps,
                learning_rate,
            )

            local_opt_improved = local_opt_fitness < updated_global_best_fitness
            updated_global_best_position = (
                local_opt_position
                if local_opt_improved
                else global_best_candidate_position
            )
            updated_global_best_fitness = (
                local_opt_fitness
                if local_opt_improved
                else updated_global_best_fitness
            )

            if local_opt_improved and (
                updated_global_best_fitness < swarm_state.global_best_fitness
            ):
                updated_personal_best_positions[global_best_candidate_idx] = (
                    updated_global_best_position
                )
                updated_personal_best_fitness[global_best_candidate_idx] = (
                    updated_global_best_fitness
                )
        else:
            updated_global_best_position = global_best_candidate_position

        updated_history = swarm_state.fitness_history
        updated_history[i] = updated_global_best_fitness

        swarm_state = SwarmState(
            positions=updated_positions,
            velocities=updated_velocities,
            personal_best_positions=updated_personal_best_positions,
            personal_best_fitness=updated_personal_best_fitness,
            global_best_position=updated_global_best_position,
            global_best_fitness=updated_global_best_fitness,
            rng=swarm_state.rng,
            fitness_history=updated_history,
        )

    return (
        swarm_state.global_best_position,
        swarm_state.global_best_fitness,
        swarm_state.fitness_history,
    )

## PSOX (JAX + JAXopt)

In [6]:
def hooke_jeeves(
    fun: callable,
    x0: jnp.ndarray,
    n_steps: int,
    lower: jnp.ndarray,
    upper: jnp.ndarray,
) -> jnp.ndarray:
    n_dims = x0.shape[0]
    initial_step = (upper - lower) * 0.1

    def exploratory_move(x: jnp.ndarray, step_size: jnp.ndarray) -> jnp.ndarray:
        def try_dim(x_curr: jnp.ndarray, dim_idx: int) -> tuple:
            e = jax.nn.one_hot(dim_idx, n_dims)
            x_plus = jnp.clip(x_curr + step_size * e, lower, upper)
            x_minus = jnp.clip(x_curr - step_size * e, lower, upper)

            f_curr = fun(x_curr)
            f_plus = fun(x_plus)
            f_minus = fun(x_minus)

            best = jnp.where(
                f_plus < jnp.minimum(f_curr, f_minus),
                x_plus,
                jnp.where(f_minus < f_curr, x_minus, x_curr),
            )
            return best, None

        x_explored, _ = lax.scan(try_dim, x, jnp.arange(n_dims))
        return x_explored

    def scan_step(carry: tuple, _: int) -> tuple:
        x_base, step_size = carry

        x_explored = exploratory_move(x_base, step_size)
        improved = fun(x_explored) < fun(x_base)

        x_pattern = jnp.clip(x_explored + (x_explored - x_base), lower, upper)
        x_pattern_explored = exploratory_move(x_pattern, step_size)

        next_x = jnp.where(
            improved,
            jnp.where(
                fun(x_pattern_explored) < fun(x_explored),
                x_pattern_explored,
                x_explored,
            ),
            x_base,
        )
        next_step = jnp.where(improved, step_size, step_size * 0.5)

        return (next_x, next_step), None

    (final_x, _), _ = lax.scan(
        scan_step,
        (x0, initial_step),
        jnp.arange(n_steps),
    )
    return final_x


def nelder_mead(
    fun: callable,
    x0: jnp.ndarray,
    n_steps: int,
    lower: jnp.ndarray,
    upper: jnp.ndarray,
    alpha: float = 1.0,
    gamma: float = 2.0,
    rho: float = 0.5,
    sigma: float = 0.5,
) -> jnp.ndarray:
    n_dims = x0.shape[0]

    step = jnp.full(n_dims, (upper - lower) * 0.05)
    simplex = jnp.vstack([x0, x0 + jnp.diag(step)])
    simplex = jnp.clip(simplex, lower, upper)

    def nm_step(simplex: jnp.ndarray, _: int) -> tuple:
        fitness = vmap(fun)(simplex)

        order = jnp.argsort(fitness)
        simplex = simplex[order]
        fitness = fitness[order]

        centroid = jnp.mean(simplex[:-1], axis=0)

        x_r = jnp.clip(centroid + alpha * (centroid - simplex[-1]), lower, upper)
        f_r = fun(x_r)

        x_e = jnp.clip(centroid + gamma * (x_r - centroid), lower, upper)
        f_e = fun(x_e)

        x_c = jnp.clip(centroid + rho * (simplex[-1] - centroid), lower, upper)
        f_c = fun(x_c)

        shrunk = jnp.clip(
            simplex[0] + sigma * (simplex - simplex[0]), lower, upper,
        )

        use_expansion = f_r < fitness[0]
        use_reflection = (f_r >= fitness[0]) & (f_r < fitness[-2])
        use_contraction = ~use_expansion & ~use_reflection
        should_shrink = use_contraction & (f_c >= fitness[-1])

        new_worst = jnp.where(
            use_expansion,
            jnp.where(f_e < f_r, x_e, x_r),
            jnp.where(
                use_reflection,
                x_r,
                jnp.where(f_c < fitness[-1], x_c, simplex[-1]),
            ),
        )

        updated_simplex = simplex.at[-1].set(new_worst) # noqa: PD008

        final_simplex = jnp.where(should_shrink, shrunk, updated_simplex)

        return final_simplex, None

    final_simplex, _ = lax.scan(nm_step, simplex, jnp.arange(n_steps))

    fitness = vmap(fun)(final_simplex)
    return final_simplex[jnp.argmin(fitness)]

@partial(
    jit,
    static_argnames=(
        "objective_function",
        "lower_bound",
        "upper_bound",
        "n_dimensions",
        "n_particles",
        "max_iterations",
        "stagnation_threshold",
        "n_local_optimization_steps",
    ),
)
def psox_jax(
    seed: random.PRNGKey,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    inertia_weight: float,
    social_coeff: float,
    cognitive_coeff: float,
    max_iterations: int,
    stagnation_threshold: int,
    n_local_optimization_steps: int,
    **_: any,
) -> tuple:
    key = seed
    lower = jnp.array(lower_bound)
    upper = jnp.array(upper_bound)
    position_key, velocity_key, state_key = random.split(key, 3)

    search_range = upper - lower
    velocity_scale = 0.1
    velocity_limit = search_range * velocity_scale

    initial_positions = random.uniform(
        position_key,
        (n_particles, n_dimensions),
        minval=lower,
        maxval=upper,
    )
    initial_velocities = random.uniform(
        velocity_key,
        (n_particles, n_dimensions),
        minval=-velocity_limit,
        maxval=velocity_limit,
    )
    initial_fitness = vmap(objective_function)(initial_positions)

    best_particle_idx = jnp.argmin(initial_fitness)
    global_best_position = initial_positions[best_particle_idx]
    global_best_fitness = initial_fitness[best_particle_idx]

    initial_state = JaxSwarmState(
        positions=initial_positions,
        velocities=initial_velocities,
        personal_best_positions=initial_positions,
        personal_best_fitness=initial_fitness,
        global_best_position=global_best_position,
        global_best_fitness=global_best_fitness,
        rng=state_key,
    )

    lbfgs_solver = jaxopt.LBFGS(
        fun=objective_function,
        maxiter=n_local_optimization_steps,
        implicit_diff=False,
    )

    def update_step(carry: tuple, _: int) -> tuple:
        swarm_state, stagnation_counter = carry

        cognitive_key, social_key, next_key = random.split(swarm_state.rng, 3)
        r1 = random.uniform(cognitive_key, (n_particles, n_dimensions))
        r2 = random.uniform(social_key, (n_particles, n_dimensions))

        inertia_term = inertia_weight * swarm_state.velocities
        cognitive_term = (
            cognitive_coeff
            * r1
            * (swarm_state.personal_best_positions - swarm_state.positions)
        )
        social_term = (
            social_coeff
            * r2
            * (swarm_state.global_best_position - swarm_state.positions)
        )

        updated_velocities = inertia_term + cognitive_term + social_term
        updated_positions = swarm_state.positions + updated_velocities
        updated_positions = jnp.clip(updated_positions, lower, upper)

        updated_fitness = vmap(objective_function)(updated_positions)

        personal_improved = updated_fitness < swarm_state.personal_best_fitness
        updated_personal_best_positions = jnp.where(
            personal_improved[:, None],
            updated_positions,
            swarm_state.personal_best_positions,
        )
        updated_personal_best_fitness = jnp.where(
            personal_improved,
            updated_fitness,
            swarm_state.personal_best_fitness,
        )

        global_best_candidate_idx = jnp.argmin(updated_personal_best_fitness)
        global_best_candidate_fitness = updated_personal_best_fitness[
            global_best_candidate_idx
        ]
        global_improved = (
            global_best_candidate_fitness < swarm_state.global_best_fitness
        )

        global_best_candidate_position = jnp.where(
            global_improved,
            updated_personal_best_positions[global_best_candidate_idx],
            swarm_state.global_best_position,
        )
        updated_global_best_fitness = jnp.where(
            global_improved,
            global_best_candidate_fitness,
            swarm_state.global_best_fitness,
        )

        def apply_local_optimization(_: None) -> tuple:
            lbfgs_position, _ = lbfgs_solver.run(global_best_candidate_position)
            hj_position = hooke_jeeves(
                objective_function,
                global_best_candidate_position,
                n_local_optimization_steps,
                lower,
                upper,
            )
            nm_position = nelder_mead(
                objective_function,
                global_best_candidate_position,
                n_local_optimization_steps,
                lower,
                upper,
            )

            lbfgs_position = jnp.clip(lbfgs_position, lower, upper)

            final_lbfgs_fitness = objective_function(lbfgs_position)
            final_hj_fitness = objective_function(hj_position)
            final_nm_fitness = objective_function(nm_position)

            best_fitness = jnp.minimum(
                jnp.minimum(final_lbfgs_fitness, final_hj_fitness),
                final_nm_fitness,
            )

            best_position = jnp.where(
                best_fitness == final_lbfgs_fitness,
                lbfgs_position,
                jnp.where(
                    best_fitness == final_hj_fitness,
                    hj_position,
                    nm_position,
                ),
            )
            return best_position, best_fitness

        def skip_local_optimization(_: None) -> tuple:
            return global_best_candidate_position, updated_global_best_fitness

        local_opt_best_position, local_opt_best_fitness = lax.cond(
            stagnation_counter >= stagnation_threshold,
            apply_local_optimization,
            skip_local_optimization,
            None,
        )

        local_opt_improved = local_opt_best_fitness < updated_global_best_fitness
        updated_global_best_position = jnp.where(
            local_opt_improved,
            local_opt_best_position,
            global_best_candidate_position,
        )
        updated_global_best_fitness = jnp.where(
            local_opt_improved,
            local_opt_best_fitness,
            updated_global_best_fitness,
        )

        any_improvement = updated_global_best_fitness < swarm_state.global_best_fitness
        winner_mask = (jnp.arange(n_particles) == global_best_candidate_idx)[:, None]
        should_update_mask = (local_opt_improved & any_improvement) & winner_mask

        final_personal_best_positions = jnp.where(
            should_update_mask,
            updated_global_best_position,
            updated_personal_best_positions,
        )
        final_personal_best_fitness = jnp.where(
            (local_opt_improved & any_improvement)
            & (jnp.arange(n_particles) == global_best_candidate_idx),
            updated_global_best_fitness,
            updated_personal_best_fitness,
        )

        next_state = JaxSwarmState(
            positions=updated_positions,
            velocities=updated_velocities,
            personal_best_positions=final_personal_best_positions,
            personal_best_fitness=final_personal_best_fitness,
            global_best_position=updated_global_best_position,
            global_best_fitness=updated_global_best_fitness,
            rng=next_key,
        )

        next_stagnation_counter = jnp.where(
            any_improvement,
            jnp.int32(0),
            stagnation_counter + jnp.int32(1),
        )

        return (next_state, next_stagnation_counter), updated_global_best_fitness

    (final_state, _), fitness_history = lax.scan(
        update_step,
        (initial_state, jnp.int32(0)),
        jnp.arange(max_iterations),
    )

    return (
        final_state.global_best_position,
        final_state.global_best_fitness,
        fitness_history,
    )

In [7]:
def parallel_psox_jax(
    seeds: random.PRNGKey,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    inertia_weight: float,
    cognitive_coeff: float,
    social_coeff: float,
    max_iterations: int,
    local_optimization_interval: int,
    n_local_optimization_steps: int,
) -> tuple[jnp.ndarray, float, jnp.ndarray]:
    psox_partial = partial(
        psox_jax,
        objective_function=objective_function,
        lower_bound=lower_bound,
        upper_bound=upper_bound,
        n_dimensions=n_dimensions,
        n_particles=n_particles,
        inertia_weight=inertia_weight,
        cognitive_coeff=cognitive_coeff,
        social_coeff=social_coeff,
        max_iterations=max_iterations,
        local_optimization_interval=local_optimization_interval,
        n_local_optimization_steps=n_local_optimization_steps,
    )

    best_positions, best_fitnesses, fitness_histories = pmap(psox_partial)(seeds)

    best_idx = jnp.argmin(best_fitnesses)
    return (
        best_positions[best_idx],
        best_fitnesses[best_idx],
        fitness_histories[best_idx],
    )

## SQPPSO

In [ ]:
def sqppso(
    seed: int,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    cognitive_coeff: float,
    social_coeff: float,
    max_iterations: int,
    w_start: float = 0.9,
    w_end: float = 0.4,
    prob_ls: float = 0.1,
    **_: any,
) -> tuple:
    np.random.seed(seed)

    if max_iterations is None:
        max_iterations = 10000 * n_dimensions // n_particles

    epsilon = 0.01 * n_dimensions / 100

    positions = np.random.uniform(
        lower_bound,
        upper_bound,
        (n_particles, n_dimensions),
    )
    velocities = np.random.uniform(
        -abs(upper_bound - lower_bound) * 0.1,
        abs(upper_bound - lower_bound) * 0.1,
        (n_particles, n_dimensions),
    )

    fitness = np.array([objective_function(p) for p in positions])

    pbest_positions = positions.copy()
    pbest_fitness = fitness.copy()

    gbest_idx = np.argmin(fitness)
    gbest_position = positions[gbest_idx].copy()
    gbest_fitness = fitness[gbest_idx]

    fitness_history = [gbest_fitness]

    ratio = 0.5

    T = 0

    current_prob_ls = prob_ls

    for iteration in range(max_iterations):
        w = w_start - (w_start - w_end) * iteration / max_iterations

        T += 1
        if n_particles // 2 == T:
            ratio = update_ratio_nras(
                positions,
                fitness,
                ratio,
                n_particles,
            )
            T = 0

        sorted_indices = np.argsort(fitness)
        n_iwpso = max(1, int(ratio * n_particles))

        iwpso_indices = sorted_indices[:n_iwpso]
        slpso_indices = sorted_indices[n_iwpso:]

        for idx in iwpso_indices:

            r1 = np.random.uniform(0, 1, n_dimensions)
            r2 = np.random.uniform(0, 1, n_dimensions)

            velocities[idx] = (
                w * velocities[idx]
                + cognitive_coeff * r1 * (pbest_positions[idx] - positions[idx])
                + social_coeff * r2 * (gbest_position - positions[idx])
            )

            positions[idx] = positions[idx] + velocities[idx]

            positions[idx] = np.clip(positions[idx], lower_bound, upper_bound)

            new_fitness = objective_function(positions[idx])
            fitness[idx] = new_fitness

            if new_fitness < pbest_fitness[idx]:
                pbest_fitness[idx] = new_fitness
                pbest_positions[idx] = positions[idx].copy()

        for idx in slpso_indices:
            rank = np.where(sorted_indices == idx)[0][0]
            lp = 1 - (rank / n_particles) * np.log(n_dimensions / 100)

            if np.random.uniform(0, 1) <= lp:
                better_particles = sorted_indices[: rank + 1] if rank > 0 else [idx]
                k_idx = np.random.choice(better_particles)

                center = np.mean(positions, axis=0)

                r1 = np.random.uniform(0, 1)
                r2 = np.random.uniform(0, 1, n_dimensions)
                r3 = np.random.uniform(0, 1, n_dimensions)

                I = positions[k_idx] - positions[idx]
                C = center - positions[idx]

                velocities[idx] = r1 * velocities[idx] + r2 * I + r3 * epsilon * C

                positions[idx] = positions[idx] + velocities[idx]

                positions[idx] = np.clip(positions[idx], lower_bound, upper_bound)

                new_fitness = objective_function(positions[idx])
                fitness[idx] = new_fitness

                if new_fitness < pbest_fitness[idx]:
                    pbest_fitness[idx] = new_fitness
                    pbest_positions[idx] = positions[idx].copy()

        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < gbest_fitness:
            gbest_fitness = fitness[current_best_idx]
            gbest_position = positions[current_best_idx].copy()

        if np.random.uniform(0, 1) <= current_prob_ls:
            improved_position, improved_fitness = apply_sqp(
                gbest_position,
                objective_function,
                lower_bound,
                upper_bound,
                n_dimensions,
            )

            if improved_fitness < gbest_fitness:
                gbest_position = improved_position
                gbest_fitness = improved_fitness
                current_prob_ls = 0.1
            else:
                current_prob_ls = 0.01

        fitness_history.append(gbest_fitness)

    return gbest_position, gbest_fitness, np.array(fitness_history)


def update_ratio_nras(positions, fitness, current_ratio, n_particles):
    sorted_indices = np.argsort(fitness)
    n_ps1 = max(1, int(current_ratio * n_particles))

    ps1_indices = sorted_indices[:n_ps1]
    ps2_indices = sorted_indices[n_ps1:]

    if len(ps2_indices) == 0:
        return current_ratio

    fitness_ps1_best = fitness[ps1_indices[0]]
    fitness_ps2_best = fitness[ps2_indices[0]]

    total_fitness = fitness_ps1_best + fitness_ps2_best
    if total_fitness == 0:
        nq1, nq2 = 0.5, 0.5
    else:
        nq1 = fitness_ps1_best / total_fitness
        nq2 = fitness_ps2_best / total_fitness

    def calculate_diversity(indices, positions, fitness):
        if len(indices) < 2:
            return 0.0
        best_pos = positions[indices[0]]
        diversity = 0.0
        for idx in indices[1:]:
            diversity += np.linalg.norm(positions[idx] - best_pos)
        return diversity

    div1 = calculate_diversity(ps1_indices, positions, fitness)
    div2 = calculate_diversity(ps2_indices, positions, fitness)

    total_div = div1 + div2
    if total_div == 0:
        nd1, nd2 = 0.5, 0.5
    else:
        nd1 = div1 / total_div
        nd2 = div2 / total_div

    k1 = (1 - nq1) + nd1
    k2 = (1 - nq2) + nd2

    total_k = k1 + k2
    if total_k == 0:
        new_ratio = 0.5
    else:
        new_ratio = k1 / total_k

    new_ratio = max(0.1, min(0.9, new_ratio))

    return new_ratio


def apply_sqp(
    current_best,
    objective_function,
    lower_bound,
    upper_bound,
    n_dimensions,
):
    bounds = [(lower_bound, upper_bound) for _ in range(n_dimensions)]

    try:
        result = minimize(
            objective_function,
            current_best,
            method="SLSQP",
            bounds=bounds,
            options={"maxiter": 100, "ftol": 1e-9},
        )

        if result.success and result.fun < objective_function(current_best):
            return result.x, result.fun
        return current_best, objective_function(current_best)
    except:
        return current_best, objective_function(current_best)

## EPSO

In [ ]:
def epso(
    seed: int,
    objective_function: callable,
    lower_bound: float,
    upper_bound: float,
    n_dimensions: int,
    n_particles: int,
    max_iterations: int,
    learning_period: int = 50,
    **_: any,
) -> tuple[np.ndarray, float, list[float]]:
    np.random.seed(seed)

    n_small = int(n_particles * 0.375)
    n_large = n_particles - n_small

    positions = np.random.uniform(lower_bound, upper_bound, (n_particles, n_dimensions))
    velocities = np.random.uniform(-1, 1, (n_particles, n_dimensions))

    fitness = np.array([objective_function(pos) for pos in positions])

    pbest_positions = positions.copy()
    pbest_fitness = fitness.copy()

    gbest_idx = np.argmin(fitness)
    gbest_position = positions[gbest_idx].copy()
    gbest_fitness = fitness[gbest_idx]

    fitness_history = [gbest_fitness]

    n_strategies = 5
    strategy_names = ["InertiaWeight", "CLPSO-gbest", "FDR-PSO", "HPSO-TVAC", "LIPS"]

    probabilities = np.ones(n_strategies) / n_strategies

    success_memory = [[] for _ in range(n_strategies)]
    failure_memory = [[] for _ in range(n_strategies)]

    pc_values = 0.05 + 0.45 * (
        np.exp(10 * np.arange(n_particles) / (n_particles - 1)) - 1
    ) / (np.exp(10) - 1)
    clpso_refreshing_gap = 7
    clpso_learn_generations = np.zeros(n_particles, dtype=int)
    clpso_exemplars = np.zeros((n_particles, n_dimensions), dtype=int)

    for i in range(n_particles):
        for d in range(n_dimensions):
            if np.random.rand() < pc_values[i]:
                idx1, idx2 = np.random.choice(n_particles, 2, replace=False)
                clpso_exemplars[i, d] = (
                    idx1 if pbest_fitness[idx1] < pbest_fitness[idx2] else idx2
                )
            else:
                clpso_exemplars[i, d] = i


    lips_nsize = 2

    for iteration in range(max_iterations):
        w = 0.9 - 0.5 * (iteration / max_iterations)
        c1_tvac = 2.5 - 2.0 * (iteration / max_iterations)
        c2_tvac = 0.5 + 2.0 * (iteration / max_iterations)
        c_clpso = 3.0 - 1.5 * (iteration / max_iterations)

        if iteration < learning_period:
            selected_strategies = stochastic_universal_sampling(probabilities, n_large)
        else:
            selected_strategies = stochastic_universal_sampling(probabilities, n_large)

        fitness_before = fitness.copy()

        for i in range(n_small):
            velocity_new, position_new = update_clpso(
                positions[i],
                velocities[i],
                pbest_positions,
                clpso_exemplars[i],
                gbest_position,
                w,
                c1_tvac,
                c2_tvac,
                lower_bound,
                upper_bound,
            )

            velocities[i] = velocity_new
            positions[i] = position_new
            fitness[i] = objective_function(positions[i])

            if fitness[i] < pbest_fitness[i]:
                pbest_positions[i] = positions[i].copy()
                pbest_fitness[i] = fitness[i]

            clpso_learn_generations[i] += 1
            if clpso_learn_generations[i] >= clpso_refreshing_gap:
                if fitness[i] >= pbest_fitness[i]:
                    clpso_learn_generations[i] = 0
                    for d in range(n_dimensions):
                        if np.random.rand() < pc_values[i]:
                            idx1, idx2 = np.random.choice(n_particles, 2, replace=False)
                            clpso_exemplars[i, d] = (
                                idx1
                                if pbest_fitness[idx1] < pbest_fitness[idx2]
                                else idx2
                            )
                        else:
                            clpso_exemplars[i, d] = i

        for idx, i in enumerate(range(n_small, n_particles)):
            strategy = selected_strategies[idx]

            if strategy == 0:
                velocity_new, position_new = update_inertia_weight_pso(
                    positions[i],
                    velocities[i],
                    pbest_positions[i],
                    gbest_position,
                    w,
                    2.0,
                    2.0,
                    lower_bound,
                    upper_bound,
                )

            elif strategy == 1:
                velocity_new, position_new = update_clpso_gbest(
                    positions[i],
                    velocities[i],
                    pbest_positions,
                    clpso_exemplars[i],
                    gbest_position,
                    w,
                    c1_tvac,
                    c2_tvac,
                    lower_bound,
                    upper_bound,
                )

            elif strategy == 2:
                velocity_new, position_new = update_fdr_pso(
                    positions[i],
                    velocities[i],
                    pbest_positions[i],
                    gbest_position,
                    positions,
                    pbest_positions,
                    fitness,
                    w,
                    lower_bound,
                    upper_bound,
                )

            elif strategy == 3:
                velocity_new, position_new = update_hpso_tvac(
                    positions[i],
                    velocities[i],
                    pbest_positions[i],
                    gbest_position,
                    c1_tvac,
                    c2_tvac,
                    lower_bound,
                    upper_bound,
                )

            else:
                velocity_new, position_new = update_lips(
                    positions[i],
                    velocities[i],
                    pbest_positions,
                    positions,
                    i,
                    lips_nsize,
                    lower_bound,
                    upper_bound,
                )

            velocities[i] = velocity_new
            positions[i] = position_new
            fitness[i] = objective_function(positions[i])

            if fitness[i] < fitness_before[i]:
                success_memory[strategy].append(1)
                failure_memory[strategy].append(0)
            else:
                success_memory[strategy].append(0)
                failure_memory[strategy].append(1)

            if len(success_memory[strategy]) > learning_period:
                success_memory[strategy].pop(0)
                failure_memory[strategy].pop(0)

            if fitness[i] < pbest_fitness[i]:
                pbest_positions[i] = positions[i].copy()
                pbest_fitness[i] = fitness[i]

        current_best_idx = np.argmin(fitness)
        if fitness[current_best_idx] < gbest_fitness:
            gbest_position = positions[current_best_idx].copy()
            gbest_fitness = fitness[current_best_idx]

        fitness_history.append(gbest_fitness)

        if iteration >= learning_period:
            probabilities = update_probabilities(
                success_memory,
                failure_memory,
                n_strategies,
            )

        if iteration % (max_iterations // 3) == 0 and lips_nsize < 5:
            lips_nsize = min(lips_nsize + 1, 5)

    return gbest_position, gbest_fitness, np.array(fitness_history)


def stochastic_universal_sampling(
    probabilities: np.ndarray,
    n_samples: int,
) -> np.ndarray:
    cumsum = np.cumsum(probabilities)
    step = 1.0 / n_samples
    start = np.random.uniform(0, step)
    pointers = start + np.arange(n_samples) * step

    selected = np.zeros(n_samples, dtype=int)
    for i, pointer in enumerate(pointers):
        selected[i] = np.searchsorted(cumsum, pointer)

    return selected


def update_probabilities(
    success_memory: list,
    failure_memory: list,
    n_strategies: int,
) -> np.ndarray:
    epsilon = 0.01
    success_rates = np.zeros(n_strategies)

    for k in range(n_strategies):
        if len(success_memory[k]) > 0:
            total_success = sum(success_memory[k])
            total_attempts = sum(success_memory[k]) + sum(failure_memory[k])
            success_rates[k] = (total_success / total_attempts) + epsilon
        else:
            success_rates[k] = epsilon

    return success_rates / np.sum(success_rates)


def update_inertia_weight_pso(
    position: np.ndarray,
    velocity: np.ndarray,
    pbest: np.ndarray,
    gbest: np.ndarray,
    w: float,
    c1: float,
    c2: float,
    lb: float,
    ub: float,
) -> tuple[np.ndarray, np.ndarray]:
    r1, r2 = np.random.rand(2, len(position))

    velocity_new = (
        w * velocity + c1 * r1 * (pbest - position) + c2 * r2 * (gbest - position)
    )

    position_new = position + velocity_new
    position_new = np.clip(position_new, lb, ub)

    return velocity_new, position_new


def update_clpso(
    position: np.ndarray,
    velocity: np.ndarray,
    pbest_all: np.ndarray,
    exemplar_indices: np.ndarray,
    gbest: np.ndarray,
    w: float,
    c1: float,
    c2: float,
    lb: float,
    ub: float,
) -> tuple[np.ndarray, np.ndarray]:
    r = np.random.rand(len(position))

    exemplar = np.array(
        [pbest_all[int(idx), d] for d, idx in enumerate(exemplar_indices)],
    )

    velocity_new = w * velocity + c1 * r * (exemplar - position)
    position_new = position + velocity_new
    position_new = np.clip(position_new, lb, ub)

    return velocity_new, position_new


def update_clpso_gbest(
    position: np.ndarray,
    velocity: np.ndarray,
    pbest_all: np.ndarray,
    exemplar_indices: np.ndarray,
    gbest: np.ndarray,
    w: float,
    c1: float,
    c2: float,
    lb: float,
    ub: float,
) -> tuple[np.ndarray, np.ndarray]:
    r1, r2 = np.random.rand(2, len(position))

    exemplar = np.array(
        [pbest_all[int(idx), d] for d, idx in enumerate(exemplar_indices)],
    )

    velocity_new = (
        w * velocity + c1 * r1 * (exemplar - position) + c2 * r2 * (gbest - position)
    )

    position_new = position + velocity_new
    position_new = np.clip(position_new, lb, ub)

    return velocity_new, position_new


def update_fdr_pso(
    position: np.ndarray,
    velocity: np.ndarray,
    pbest: np.ndarray,
    gbest: np.ndarray,
    all_positions: np.ndarray,
    all_pbest: np.ndarray,
    all_fitness: np.ndarray,
    w: float,
    lb: float,
    ub: float,
) -> tuple[np.ndarray, np.ndarray]:
    c1, c2, c3 = 1.0, 1.0, 2.0

    nbest = find_nbest_fdr(position, all_positions, all_pbest, all_fitness)

    r1, r2 = np.random.rand(2, len(position))

    velocity_new = (
        w * velocity
        + c1 * r1 * (pbest - position)
        + c2 * r2 * (gbest - position)
        + c3 * (nbest - position)
    )

    position_new = position + velocity_new
    position_new = np.clip(position_new, lb, ub)

    return velocity_new, position_new


def find_nbest_fdr(
    position: np.ndarray,
    all_positions: np.ndarray,
    all_pbest: np.ndarray,
    all_fitness: np.ndarray,
) -> np.ndarray:
    best_ratio = -np.inf
    best_pbest = all_pbest[0]

    for i in range(len(all_positions)):
        distance = np.mean(np.abs(all_pbest[i] - position))

        if distance > 1e-10:
            ratio = -all_fitness[i] / distance

            if ratio > best_ratio:
                best_ratio = ratio
                best_pbest = all_pbest[i]

    return best_pbest


def update_hpso_tvac(
    position: np.ndarray,
    velocity: np.ndarray,
    pbest: np.ndarray,
    gbest: np.ndarray,
    c1: float,
    c2: float,
    lb: float,
    ub: float,
) -> tuple[np.ndarray, np.ndarray]:
    r1, r2 = np.random.rand(2, len(position))

    velocity_new = c1 * r1 * (pbest - position) + c2 * r2 * (gbest - position)

    if np.allclose(velocity_new, 0):
        vmax = 0.2 * (ub - lb)
        velocity_new = np.random.uniform(-vmax, vmax, len(position))

    position_new = position + velocity_new
    position_new = np.clip(position_new, lb, ub)

    return velocity_new, position_new


def update_lips(
    position: np.ndarray,
    velocity: np.ndarray,
    all_pbest: np.ndarray,
    all_positions: np.ndarray,
    current_idx: int,
    nsize: int,
    lb: float,
    ub: float,
) -> tuple[np.ndarray, np.ndarray]:
    chi = 0.729


    distances = np.sum((all_pbest - all_pbest[current_idx]) ** 2, axis=1)
    neighbor_indices = np.argsort(distances)[
        1 : nsize + 1
    ]

    phi_j = np.random.uniform(0, 4.1 / nsize, nsize)
    phi_sum = np.sum(phi_j)

    P_i = (
        np.sum(
            [phi_j[j] * all_pbest[neighbor_indices[j]] for j in range(nsize)],
            axis=0,
        )
        / phi_sum
    )

    velocity_new = chi * (velocity + phi_sum * (P_i - position))
    position_new = position + velocity_new
    position_new = np.clip(position_new, lb, ub)

    return velocity_new, position_new

# Benchmark Functions

In [10]:
def _get_xp(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    return jnp if isinstance(x, jax.Array) else np


def ackley(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    n = x.shape[0]
    sum_sq = xp.sum(x**2)
    sum_cos = xp.sum(xp.cos(2 * xp.pi * x))
    return (
        -20 * xp.exp(-0.2 * xp.sqrt(sum_sq / n))
        - xp.exp(sum_cos / n)
        + 20
        + xp.e
    )


def cigar(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    return xp.sum(x**2) + 1e6 * xp.sum(x[1:] ** 2)


def griewank(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    n = x.shape[0]
    i = xp.arange(1, n + 1)
    return xp.sum(x**2) / 4000 - xp.prod(xp.cos(x / xp.sqrt(i))) + 1


def levy(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    w = 1 + (x - 1) / 4
    term1 = xp.sin(xp.pi * w[0]) ** 2
    term2 = xp.sum((w[:-1] - 1) ** 2 * (1 + 10 * xp.sin(xp.pi * w[:-1] + 1) ** 2))
    term3 = (w[-1] - 1) ** 2 * (1 + xp.sin(2 * xp.pi * w[-1]) ** 2)
    return term1 + term2 + term3


def rastrigin(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    n = x.shape[0]
    return 10 * n + xp.sum(x**2 - 10 * xp.cos(2 * xp.pi * x))


def rosenbrock(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    return xp.sum(100 * (x[1:] - x[:-1] ** 2) ** 2 + (1 - x[:-1]) ** 2)


def schwefel(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    n = x.shape[0]
    return 418.9829 * n - xp.sum(x * xp.sin(xp.sqrt(xp.abs(x))))


def sphere(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    return xp.sum(x**2)


def zakharov(x: np.ndarray | jax.Array) -> np.ndarray | jax.Array:
    xp = _get_xp(x)
    n = x.shape[0]
    i = xp.arange(1, n + 1)
    s = xp.sum(0.5 * i * x)
    return xp.sum(x**2) + s**2 + s**4

# Configuration

In [ ]:
parameters = {
    "n_particles": 85,
    "max_iterations": 100,
    "cognitive_coeff": 1.5,
    "social_coeff": 1.5,
    "inertia_weight": 0.7,
}

extra_parameters = {
    "stagnation_threshold": 5,
    "local_optimization_interval": 10,
    "n_local_optimization_steps": 5,
}

algorithms = [
    Algorithm("PSO", pso_jax, parameters),
    Algorithm("EPSO", epso, {**parameters, **parameters}),
    Algorithm("SQPPSO", sqppso, {**parameters, **parameters}),
    Algorithm("DRS-PSO", psox_jax, {**parameters, **extra_parameters}),
]

benchmarks = [
    Benchmark("Ackley", ackley, -32.768, 32.768),
    Benchmark("Cigar", cigar, -100, 100),
    Benchmark("Griewank", griewank, -600, 600),
    Benchmark("Rastrigin", rastrigin, -5.12, 5.12),
    Benchmark("Rosenbrock", rosenbrock, -5, 10),
    Benchmark("Schwefel", schwefel, -500, 500),
    Benchmark("Sphere", sphere, -5.12, 5.12),
    Benchmark("Zakharov", zakharov, -5.0, 10.0),
]

dimensions = [30, 100, 300, 500]

n_runs = 50

# Experiments

In [12]:
reproducible = True

In [13]:
def run_experiments(
    algorithms: list[Algorithm],
    benchmarks: list[Benchmark],
    dimensions: list[int],
    n_runs: int,
) -> list[Result]:
    results = []
    for algorithm, dimension, benchmark, run in product(
        algorithms,
        dimensions,
        benchmarks,
        range(1, n_runs + 1),
    ):
        seed_value = run if reproducible else int(time.time())

        print(f"{algorithm.name} | {benchmark.name} | {dimension} | {run}/{n_runs}")

        if algorithm.name == "Parallel PSOX (JAX)":
            seed = random.split(
                random.PRNGKey(seed_value),
                jax.local_device_count(),
            )

            algorithm.function(
                seed,
                benchmark.function,
                benchmark.lower_bound,
                benchmark.upper_bound,
                dimension,
                algorithm.parameters["n_particles"],
                algorithm.parameters["inertia_weight"],
                algorithm.parameters["cognitive_coeff"],
                algorithm.parameters["social_coeff"],
                algorithm.parameters["max_iterations"],
                algorithm.parameters["local_optimization_interval"],
                algorithm.parameters["n_local_optimization_steps"],
            )

            start = time.perf_counter()

            result = algorithm.function(
                seed,
                benchmark.function,
                benchmark.lower_bound,
                benchmark.upper_bound,
                dimension,
                algorithm.parameters["n_particles"],
                algorithm.parameters["inertia_weight"],
                algorithm.parameters["cognitive_coeff"],
                algorithm.parameters["social_coeff"],
                algorithm.parameters["max_iterations"],
                algorithm.parameters["local_optimization_interval"],
                algorithm.parameters["n_local_optimization_steps"],
            )
            block_until_ready(result)

            position, fitness, history = result

        elif algorithm.name in ["PSO", "DRS-PSO", "SQPPSO"]:
            seed = random.PRNGKey(seed_value)

            algorithm.function(
                seed,
                benchmark.function,
                benchmark.lower_bound,
                benchmark.upper_bound,
                dimension,
                **algorithm.parameters,
            )

            start = time.perf_counter()

            result = algorithm.function(
                seed,
                benchmark.function,
                benchmark.lower_bound,
                benchmark.upper_bound,
                dimension,
                **algorithm.parameters,
            )
            block_until_ready(result)

            position, fitness, history = result

        else:
            seed = seed_value

            start = time.perf_counter()

            position, fitness, history = algorithm.function(
                seed,
                benchmark.function,
                benchmark.lower_bound,
                benchmark.upper_bound,
                dimension,
                **algorithm.parameters,
            )

        end = time.perf_counter()

        results.append(
            Result(
                dimension=dimension,
                benchmark=benchmark.name,
                algorithm=algorithm.name,
                best_fitness=float(fitness),
                best_position=position.tolist(),
                fitness_history=history.tolist(),
                execution_time=end - start,
            ),
        )

    return results

In [14]:
results = run_experiments(algorithms, benchmarks, dimensions, n_runs)
pd.DataFrame(results).to_csv("../data/experiments_results.csv", index=False)

PSO | Ackley | 30 | 1/50
PSO | Ackley | 30 | 2/50
PSO | Ackley | 30 | 3/50
PSO | Ackley | 30 | 4/50
PSO | Ackley | 30 | 5/50
PSO | Ackley | 30 | 6/50
PSO | Ackley | 30 | 7/50
PSO | Ackley | 30 | 8/50
PSO | Ackley | 30 | 9/50
PSO | Ackley | 30 | 10/50
PSO | Ackley | 30 | 11/50
PSO | Ackley | 30 | 12/50
PSO | Ackley | 30 | 13/50
PSO | Ackley | 30 | 14/50
PSO | Ackley | 30 | 15/50
PSO | Ackley | 30 | 16/50
PSO | Ackley | 30 | 17/50
PSO | Ackley | 30 | 18/50
PSO | Ackley | 30 | 19/50
PSO | Ackley | 30 | 20/50
PSO | Ackley | 30 | 21/50
PSO | Ackley | 30 | 22/50
PSO | Ackley | 30 | 23/50
PSO | Ackley | 30 | 24/50
PSO | Ackley | 30 | 25/50
PSO | Ackley | 30 | 26/50
PSO | Ackley | 30 | 27/50
PSO | Ackley | 30 | 28/50
PSO | Ackley | 30 | 29/50
PSO | Ackley | 30 | 30/50
PSO | Ackley | 30 | 31/50
PSO | Ackley | 30 | 32/50
PSO | Ackley | 30 | 33/50
PSO | Ackley | 30 | 34/50
PSO | Ackley | 30 | 35/50
PSO | Ackley | 30 | 36/50
PSO | Ackley | 30 | 37/50
PSO | Ackley | 30 | 38/50
PSO | Ackley | 30 | 3